# Interior Design Chatbot - Fine-tuning with QLoRA

Base model: `Qwen/Qwen2.5-1.5B-Instruct`  
Method: QLoRA (4-bit + LoRA)  
Dataset: `disham993/Synthetic_Furniture_Dataset` (HuggingFace) + custom QA pairs  
Libraries: transformers, peft, trl, bitsandbytes, datasets

Make sure to set runtime to T4 GPU on Colab

## Install dependencies

In [ ]:
!pip install transformers peft trl bitsandbytes datasets accelerate ipywidgets -q
!pip install pyarrow --upgrade -q
# need to restart runtime after this

## Imports and GPU check

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset
import ipywidgets as widgets
from datasets import Dataset
from IPython.display import display, HTML, clear_output
from trl import SFTTrainer, SFTConfig

# check if gpu is available
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU detected - switch to T4 GPU in runtime settings")

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Load and prepare the dataset

In [ ]:
DATASET_NAME = "disham993/Synthetic_Furniture_Dataset"

print(f"Loading dataset: {DATASET_NAME}")
raw = load_dataset(DATASET_NAME, split="train")
print(f"Raw size: {len(raw)} rows")
print(f"Columns: {raw.column_names}")
print()
print("Sample row:")
print(raw[0])

# the dataset has name, description, ad columns
# we want to convert into instruction/response format for fine-tuning
def make_qa(sample):
    instruction = (
        f"I am looking for a {sample['name']}. "
        f"What materials and textures should I look for, and how should it look and feel?"
    )
    response = (
        f"{sample['description']} {sample['ad']}"
    ).strip()
    return {"instruction": instruction, "response": response}

raw_dataset = raw.map(make_qa, remove_columns=raw.column_names)

print(f"\nDataset size after formatting: {len(raw_dataset)} samples")
print(f"Columns: {raw_dataset.column_names}")
print()
print("Formatted sample:")
print("Q:", raw_dataset[0]["instruction"])
print("A:", raw_dataset[0]["response"])

# also adding some custom QA pairs to supplement the dataset
# these cover topics the furniture dataset doesnt have (colour theory, layout, lighting etc)
from datasets import Dataset

qa_pairs = [
    # colour
    {"instruction": "What colors work best in a small living room?",
     "response": "Light, neutral tones like soft white, cream, or pale grey make small rooms feel larger by reflecting light. Add depth with a single accent wall in a muted sage or dusty blue. Avoid dark colours on all four walls as they absorb light and shrink the perceived space."},
    {"instruction": "How do I choose a colour palette for an open-plan space?",
     "response": "Use a consistent base colour (e.g. warm white or greige) across all zones to create flow. Differentiate areas with accent colours - a muted teal for the dining zone, soft terracotta for the lounge. Limit your palette to 3 colours: one dominant, one secondary, one accent."},
    {"instruction": "What paint colours make a bedroom feel calm and relaxing?",
     "response": "Soft blues, muted greens, and warm lavenders promote relaxation. Benjamin Moore's 'Tranquility' or Farrow & Ball's 'Pavilion Blue' are good choices. Pair with warm wood tones and soft white bedding to balance the coolness."},

    # layout / space planning
    {"instruction": "How should I arrange furniture in a long narrow living room?",
     "response": "Place a sofa perpendicular to the long wall to break the tunnel effect. Use a round coffee table to soften the geometry. Create two zones: a seating area near one end and a reading nook or console near the other. Keep pathways at least 75 cm wide."},
    {"instruction": "What is the best layout for a small studio apartment?",
     "response": "Use a low bookshelf or open shelving unit as a room divider between the sleeping and living zones. Place the bed against the far wall, sofa facing away from it. A fold-down desk or wall-mounted drop-leaf table saves floor space. Mirrors on one wall double the perceived depth."},

    # lighting
    {"instruction": "What lighting should I use in a home office?",
     "response": "Layer three types: ambient (ceiling flush mount or pendant), task (adjustable desk lamp, 4000-5000 K daylight), and accent (a warm LED strip behind the monitor to reduce eye strain). Position the desk so natural light comes from the side, not behind the screen."},
    {"instruction": "How do I use lighting to make a room feel cozy?",
     "response": "Use warm-white bulbs (2700 K) and place light sources at different heights: table lamps, floor lamps, and candle-level lighting. Avoid a single overhead light. Dimmer switches let you adjust intensity. Uplighting on a textured wall adds warmth without glare."},

    # materials and textures
    {"instruction": "How do I mix different wood tones in one room without clashing?",
     "response": "Pick one dominant wood tone and use it for the largest piece (e.g. oak dining table). Add one contrasting tone (e.g. walnut side table) and keep a third as an accent (e.g. rattan basket). Repeat each tone at least twice in the room. A neutral rug ties them together."},
    {"instruction": "What textiles add warmth to a minimalist room?",
     "response": "Bouclé or chunky knit throws on a linen sofa add tactile interest. Layer a wool or jute rug over hard floors. Use linen curtains in an undyed oatmeal tone. Velvet cushion covers in muted earth tones (terracotta, olive) give subtle richness without clutter."},

    # design styles
    {"instruction": "What defines Scandinavian interior design?",
     "response": "Scandinavian design prioritises function, natural light, and organic materials. Key elements: light-toned woods (birch, ash, pine), a muted colour palette with white walls, clean-lined furniture, and cozy textiles like sheepskin and knit throws - the concept of 'hygge'."},
    {"instruction": "How do I create a Japanese-inspired bedroom?",
     "response": "Start with a low platform bed or futon on a tatami-style mat. Use natural materials: hinoki cypress, bamboo, washi paper. Keep surfaces uncluttered; storage should be hidden. Earthy neutrals (warm grey, sand, charcoal) with one accent like indigo. Add a single bonsai or ikebana arrangement as a focal point. Shoji screens filter light softly."},
    {"instruction": "What are the characteristics of Mediterranean interior design?",
     "response": "Mediterranean style features terracotta tile floors, arched doorways, and textured stucco walls. The palette centres on warm earth tones - ochre, sienna, olive - with accents of cobalt blue. Wrought-iron light fixtures, ceramic hand-painted tiles, and heavy linen or cotton drapes are typical. Furniture is solid wood, often distressed."},

    # practical / budget
    {"instruction": "How can I make a rented apartment look better without permanent changes?",
     "response": "Use removable wallpaper or large-format art to cover bland walls. Swap out curtain rods (save the originals) for a style upgrade. Layer rugs over existing flooring. Use peel-and-stick tile on a kitchen backsplash. Command strips hold shelves and frames without holes."},
    {"instruction": "What are some affordable ways to refresh a living room?",
     "response": "New cushion covers and a throw change the mood for under $50. A large indoor plant like a fiddle-leaf fig adds life. Rearranging existing furniture costs nothing and can feel transformative. Paint one accent wall in a bold colour. Replace old lamp shades with modern drum or rattan shades."},
]

custom_ds = Dataset.from_list(qa_pairs)
raw_dataset = raw_dataset.select(range(len(raw_dataset)))  # keep all original
from datasets import concatenate_datasets
raw_dataset = concatenate_datasets([raw_dataset, custom_ds])
print(f"\nFinal dataset size (with custom pairs): {len(raw_dataset)}")

In [ ]:
# format into Qwen chat template for SFT training
from transformers import AutoTokenizer as _AT
tokenizer_ref = _AT.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

SYSTEM_PROMPT = (
    "You are a knowledgeable and friendly interior design consultant. "
    "Give practical, actionable advice about space planning, colour palettes, "
    "lighting, furniture arrangement, and design styles."
)

def format_sample(sample):
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": sample["instruction"]},
        {"role": "assistant", "content": sample["response"]},
    ]
    return {"text": tokenizer_ref.apply_chat_template(messages, tokenize=False)}

dataset = raw_dataset.map(format_sample, remove_columns=raw_dataset.column_names)

# 95/5 train/val split
dataset = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = dataset["train"]
eval_dataset  = dataset["test"]

print(f"Train: {len(train_dataset)} samples")
print(f"Eval:  {len(eval_dataset)} samples")
print()
print("Sample (first 500 chars):")
print(train_dataset[0]["text"][:500], "...")

## Load base model in 4-bit (QLoRA)

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "./qwen-interior-design-qlora"

# 4-bit quantisation config for QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,       # saves ~0.4 GB extra
)

print(f"Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Loading model (4-bit): {MODEL_ID}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False   # need to disable this during training

if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Model loaded - VRAM: {used:.1f} / {total:.1f} GB")

## Configure LoRA adapters

In [ ]:
# target attention + MLP layers in Qwen2.5
# tried just q_proj/v_proj first but results were better with all projection layers
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Training

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,     # effective batch size = 4*4 = 16
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    fp16=False,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=5,
    save_strategy="steps",
    save_steps=5,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",
    optim="paged_adamw_8bit",
    dataset_text_field="text",
    max_length=1024,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

# sanity check - print a sample
print(train_dataset[0]["text"][:200])
print("...")
print()

print("Starting training...")
trainer.train()
print("Done!")

## Save the adapter weights

In [ ]:
# only saving the LoRA adapter (few MB), not the full model
ADAPTER_DIR = f"{OUTPUT_DIR}/final-adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to: {ADAPTER_DIR}")

## Test the fine-tuned model

In [ ]:
# load the fine-tuned model back
from peft import AutoPeftModelForCausalLM

ft_model = AutoPeftModelForCausalLM.from_pretrained(
    ADAPTER_DIR,
    torch_dtype=torch.float16,
    device_map="auto",
    quantization_config=bnb_config,
)
ft_model.eval()
print("Fine-tuned model loaded")

In [ ]:
def generate(user_question, max_new_tokens=512):
    messages = [
        {"role": "system",  "content": SYSTEM_PROMPT},
        {"role": "user",    "content": user_question},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(ft_model.device)

    with torch.no_grad():
        output_ids = ft_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# try some questions
questions = [
    "What colors work best for a small living room?",
    "How do I create a Scandinavian-style bedroom on a budget?",
    "What lighting should I use for a home office?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {generate(q)}")
    print("-" * 50)

## Chat UI (optional)

Some questions to test with:
- "What colors work well in a small living room?"
- "How do I arrange furniture in an open-plan space?"
- "What are the key principles of Scandinavian interior design?"
- "How can I add warmth to a room without repainting?"
- "What lighting should I use for a home office?"

## VLM Edit Plan Analysis

This part takes the output from the VLM (vision language model) component and uses our fine-tuned model to review and improve the suggested edits for each object in the room.

In [ ]:
import json

# sample VLM output to test with
vlm_output = {
  "image_path": "uploads/365Cupperserangoonroad_11.jpg",
  "user_prompt": "I want a Japanese themed house",
  "vision_output": {
    "objects": ["desk","chair","bed","window","track lighting","shelving unit","plant"],
    "room_type": "bedroom",
    "scene_style": "modern",
    "object_notes": {
      "desk": "long wooden desk",
      "chair": "simple beige chair",
      "bed": "white bed with striped blanket",
      "window": "large window with blinds",
      "track lighting": "black track lighting fixtures",
      "shelving unit": "wooden shelving unit on wall",
      "plant": "potted plant on windowsill"
    }
  },
  "edit_plan": {
    "global_style": "Japanese themed house",
    "edits": [
      {"object": "desk",          "description": "a traditional Japanese low table with a natural wood finish"},
      {"object": "chair",         "description": "a zabuton cushion for sitting on the floor"},
      {"object": "bed",           "description": "a futon mattress with a woven cover"},
      {"object": "window",        "description": "paper shoji screens instead of blinds"},
      {"object": "track lighting","description": "paper lanterns hanging from the ceiling"},
      {"object": "shelving unit", "description": "a built-in bamboo shelf unit"},
      {"object": "plant",         "description": "a bonsai tree in a traditional pot"}
    ]
  }
}

def build_analysis_prompt(vlm):
    """Build a prompt that asks the model to review each proposed edit."""
    vout   = vlm["vision_output"]
    eplan  = vlm["edit_plan"]
    prompt = vlm["user_prompt"]

    edit_map = {e["object"]: e["description"] for e in eplan["edits"]}
    objects_block = "\n".join(
        f"  - {obj} | currently: {desc} | proposed edit: {edit_map.get(obj, 'no change')}"
        for obj, desc in vout["object_notes"].items()
    )

    return (
        f"A {vout['room_type']} with a {vout['scene_style']} style contains:\n{objects_block}\n\n"
        f"The user wants: \"{prompt}\"\n\n"
        f"You are a specialist in {eplan['global_style']} interior design. "
        f"For EACH object listed, provide:\n\n"
        f"OBJECT: [name]\n"
        f"  Material: [exact material e.g. solid oak, brushed steel]\n"
        f"  Texture: [exact texture e.g. matte, hand-scraped, woven]\n"
        f"  Colour: [specific colour tone]\n"
        f"  Improved suggestion: [your alternative]\n"
        f"  Why it works: [reason tied to {eplan['global_style']} design]\n\n"
        f"After all objects, add:\n"
        f"MISSING ELEMENTS: List 3-5 items not yet mentioned that would complete the look.\n\n"
        f"Be specific with materials - avoid vague terms like 'natural materials'."
    )

# run the analysis
query = build_analysis_prompt(vlm_output)
messages = [
    {"role": "system",  "content": SYSTEM_PROMPT},
    {"role": "user",    "content": query},
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors="pt").to(ft_model.device)

with torch.no_grad():
    output_ids = ft_model.generate(
        **inputs, max_new_tokens=512, temperature=0.7, do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

new_tokens = output_ids[0][inputs.input_ids.shape[1]:]
result = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print(f"Style: {vlm_output['edit_plan']['global_style']}")
print(f"Room: {vlm_output['vision_output']['room_type']}")
print("=" * 50)
print()
print(result)

In [ ]:
import time, re, statistics, json
from datetime import datetime
from pathlib import Path

# base room setup - same for all style tests
BASE_ROOM = {
    "objects": ["desk","chair","bed","window","track lighting","shelving unit","plant"],
    "room_type": "bedroom",
    "scene_style": "modern",
    "object_notes": {
        "desk":           "long wooden desk",
        "chair":          "simple beige chair",
        "bed":            "white bed with striped blanket",
        "window":         "large window with blinds",
        "track lighting": "black track lighting fixtures",
        "shelving unit":  "wooden shelving unit on wall",
        "plant":          "potted plant on windowsill",
    }
}

# different styles to test against
DESIGNS = {
    "japanese": {
        "prompt": "I want a Japanese themed bedroom",
        "edits": [
            {"object": "desk",           "description": "a traditional low chabudai table with a natural wood finish"},
            {"object": "chair",          "description": "a zabuton floor cushion for seated work"},
            {"object": "bed",            "description": "a futon mattress with a woven indigo cover"},
            {"object": "window",         "description": "paper shoji screens replacing the blinds"},
            {"object": "track lighting", "description": "paper lanterns suspended from the ceiling"},
            {"object": "shelving unit",  "description": "a built-in bamboo shelf unit"},
            {"object": "plant",          "description": "a bonsai tree in a glazed ceramic pot"},
        ]
    },
    "victorian": {
        "prompt": "I want a Victorian style bedroom",
        "edits": [
            {"object": "desk",           "description": "an ornate writing desk with carved dark mahogany legs"},
            {"object": "chair",          "description": "a tufted wingback chair in deep burgundy velvet"},
            {"object": "bed",            "description": "a four-poster bed with heavy damask drapes"},
            {"object": "window",         "description": "floor-length velvet curtains with brass tiebacks"},
            {"object": "track lighting", "description": "a brass chandelier with frosted glass globe shades"},
            {"object": "shelving unit",  "description": "a dark walnut bookcase with glass doors and brass hardware"},
            {"object": "plant",          "description": "a large fern in a brass jardiniere stand"},
        ]
    },
    "modern": {
        "prompt": "I want a modern minimalist bedroom",
        "edits": [
            {"object": "desk",           "description": "a wall-mounted floating desk in white lacquer"},
            {"object": "chair",          "description": "a moulded shell chair on slim hairpin legs"},
            {"object": "bed",            "description": "a low platform bed with an upholstered light grey headboard"},
            {"object": "window",         "description": "floor-to-ceiling roller blinds in warm white"},
            {"object": "track lighting", "description": "recessed LED spotlights flush to the ceiling"},
            {"object": "shelving unit",  "description": "a modular open shelving system in white powder-coated steel"},
            {"object": "plant",          "description": "a single sculptural cactus in a concrete planter"},
        ]
    },
    "scandinavian": {
        "prompt": "I want a Scandinavian hygge bedroom",
        "edits": [
            {"object": "desk",           "description": "a light birch desk with tapered solid wood legs"},
            {"object": "chair",          "description": "a sheepskin-draped solid ash stool"},
            {"object": "bed",            "description": "a low ash frame with a white cotton duvet and knit throw"},
            {"object": "window",         "description": "sheer undyed linen curtains that pool softly on the floor"},
            {"object": "track lighting", "description": "a white paper pendant lamp on a fabric cord"},
            {"object": "shelving unit",  "description": "a floating light pine shelf with minimal objects"},
            {"object": "plant",          "description": "eucalyptus stems in a simple matte ceramic vase"},
        ]
    },
    "industrial": {
        "prompt": "I want an industrial loft bedroom",
        "edits": [
            {"object": "desk",           "description": "a reclaimed oak plank desk on black iron pipe legs"},
            {"object": "chair",          "description": "a metal factory stool with a worn leather seat pad"},
            {"object": "bed",            "description": "a black powder-coated steel bed frame with visible hex bolts"},
            {"object": "window",         "description": "steel-framed windows left bare to maximise light"},
            {"object": "track lighting", "description": "bare Edison bulb pendants on black iron conduit pipe"},
            {"object": "shelving unit",  "description": "iron pipe brackets holding raw-edge reclaimed wood planks"},
            {"object": "plant",          "description": "a tall snake plant in a galvanised steel bucket"},
        ]
    },
    "bohemian": {
        "prompt": "I want a bohemian eclectic bedroom",
        "edits": [
            {"object": "desk",           "description": "a distressed pine desk with a macrame-trimmed drawer pull"},
            {"object": "chair",          "description": "a rattan hanging egg chair with layered kilim cushions"},
            {"object": "bed",            "description": "a low divan layered with patterned textiles and throw pillows"},
            {"object": "window",         "description": "sheer embroidered muslin curtains with tassel trim"},
            {"object": "track lighting", "description": "warm string fairy lights and a woven rattan pendant"},
            {"object": "shelving unit",  "description": "mismatched reclaimed wood floating shelves with trailing plants"},
            {"object": "plant",          "description": "trailing pothos in a hanging macrame plant holder"},
        ]
    },
}

# scoring functions

# words that are too generic - we want the model to be specific
GENERIC_TERMS = [
    "natural wood", "soft fabric", "traditional style", "traditional material",
    "wooden material", "nice", "elegant", "classic",
]

# patterns that indicate the model gave specific material/texture names
SPECIFIC_PATTERNS = [
    r"\b(hinoki|sugi|cedar|cypress|paulownia|kiri|persimmon|mahogany|walnut|birch|ash|oak|pine|teak)\b",
    r"\b(ramie|habotai|asa|hemp|linen|washi|kozo|velvet|damask|muslin|kilim|cotton|sheepskin)\b",
    r"\b(urushi|lacquer|tung.oil|beeswax|shou.sugi|charred|powder.coated|galvanised)\b",
    r"\b(shigaraki|bizen|stoneware|brass|iron|steel)\b",
    r"\d+\s*(cm|mm|inches?)",
    r"#[0-9a-fA-F]{6}",
]

def score_completeness(output, vlm):
    """Check how many objects from the edit plan the model actually addressed."""
    objects = [e["object"].lower() for e in vlm["edit_plan"]["edits"]]
    covered = [o for o in objects if o in output.lower()]
    return {
        "score": round(len(covered) / len(objects), 3),
        "covered": covered,
        "missing": [o for o in objects if o not in covered],
    }

def score_specificity(output):
    """Ratio of specific material names vs generic terms."""
    text = output.lower()
    generic_hits = sum(1 for t in GENERIC_TERMS if t in text)
    specific_hits = 0
    found = []
    for pat in SPECIFIC_PATTERNS:
        matches = re.findall(pat, text)
        specific_hits += len(matches)
        found.extend(matches)
    total = specific_hits + generic_hits
    return {
        "score": round(specific_hits / total, 3) if total > 0 else 0.0,
        "specific_count": specific_hits,
        "generic_count": generic_hits,
        "found": list(set(found))[:8],
    }

def run_once(vlm):
    """Run the model once on the VLM output and measure time."""
    prompt = build_analysis_prompt(vlm)
    messages = [
        {"role": "system",  "content": SYSTEM_PROMPT},
        {"role": "user",    "content": prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(ft_model.device)

    t0 = time.time()
    with torch.no_grad():
        output_ids = ft_model.generate(
            **inputs, max_new_tokens=512, temperature=0.7, do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.time() - t0

    new_tokens = output_ids[0][inputs.input_ids.shape[1]:]
    decoded = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    n_tok = len(decoded.split())
    return decoded, elapsed, n_tok

# run the benchmark - 3 runs per style
N_RUNS = 3
all_results = {}

for style, cfg in DESIGNS.items():
    vlm = {
        "image_path":    "uploads/test_room.jpg",
        "user_prompt":   cfg["prompt"],
        "vision_output": BASE_ROOM,
        "edit_plan":     {"global_style": f"{style} style", "edits": cfg["edits"]},
    }

    print(f"\n{'='*50}")
    print(f"  {style.upper()} ({N_RUNS} runs)")
    print(f"{'='*50}")

    runs = []
    for i in range(N_RUNS):
        print(f"  run {i+1}/{N_RUNS} ...", end=" ", flush=True)
        out, elapsed, n_tok = run_once(vlm)
        c = score_completeness(out, vlm)
        s = score_specificity(out)
        print(f"{elapsed:.1f}s | comp={c['score']*100:.0f}% | spec={s['score']*100:.0f}%")
        runs.append({
            "run": i + 1,
            "output": out,
            "completeness": c,
            "specificity": s,
            "speed": {"elapsed_s": round(elapsed, 2), "tokens": n_tok,
                      "tok_per_s": round(n_tok / elapsed, 1)},
        })

    c_vals = [r["completeness"]["score"] for r in runs]
    s_vals = [r["specificity"]["score"]  for r in runs]
    t_vals = [r["speed"]["elapsed_s"]    for r in runs]

    all_results[style] = {
        "runs": runs,
        "summary": {
            "completeness_mean":  round(statistics.mean(c_vals), 3),
            "completeness_stdev": round(statistics.stdev(c_vals), 3),
            "specificity_mean":   round(statistics.mean(s_vals), 3),
            "specificity_stdev":  round(statistics.stdev(s_vals), 3),
            "speed_mean_s":       round(statistics.mean(t_vals), 2),
            "speed_stdev_s":      round(statistics.stdev(t_vals), 2),
        }
    }

# save results
out_path = Path("benchmark_results.json")
out_path.write_text(json.dumps({
    "timestamp":        datetime.now().isoformat(),
    "model":            MODEL_ID,
    "n_runs_per_style": N_RUNS,
    "results":          all_results,
}, indent=2))
print(f"\nResults saved to {out_path}")

# save markdown report
md_lines = [
    f"# Benchmark Report\n",
    f"**Model:** {MODEL_ID}  |  **Runs per style:** {N_RUNS}  |  "
    f"**Date:** {datetime.now().strftime('%Y-%m-%d %H:%M')}\n",
    "| Style | Comp mean | Comp +/- | Spec mean | Spec +/- | Speed (s) | Speed +/- |",
    "|---|---|---|---|---|---|---|",
]
for style, data in all_results.items():
    s = data["summary"]
    md_lines.append(
        f"| {style} | {s['completeness_mean']:.2f} | {s['completeness_stdev']:.3f} "
        f"| {s['specificity_mean']:.2f} | {s['specificity_stdev']:.3f} "
        f"| {s['speed_mean_s']:.1f} | {s['speed_stdev_s']:.2f} |"
    )

md_lines += ["\n## Detailed Outputs\n"]
for style, data in all_results.items():
    md_lines.append(f"### {style.title()}")
    for r in data["runs"]:
        md_lines.append(f"\n**Run {r['run']}** - "
                        f"comp={r['completeness']['score']*100:.0f}% | "
                        f"spec={r['specificity']['score']*100:.0f}% | "
                        f"{r['speed']['elapsed_s']}s\n")
        md_lines.append(r["output"])
        md_lines.append("\n---")

Path("benchmark_report.md").write_text("\n".join(md_lines))
print("Report saved to benchmark_report.md")

# print summary
print(f"\n{'Style':<15} {'Comp':>6} {'+-':>6} {'Spec':>6} {'+-':>6} {'Speed':>8} {'+-':>6}")
print("-" * 55)
for style, data in all_results.items():
    s = data["summary"]
    print(f"{style:<15} {s['completeness_mean']:>6.2f} {s['completeness_stdev']:>6.3f} "
          f"{s['specificity_mean']:>6.2f} {s['specificity_stdev']:>6.3f} "
          f"{s['speed_mean_s']:>8.1f} {s['speed_stdev_s']:>6.2f}")